In [ ]:
import os
import cv2 as cv

In [14]:
def parse_annotations(annotations_file):
    """
    Parse BelgiumTS sequence annotation file.
    Returns dictionary: {'camera_id/image.frame_num.jp2': [{'bbox': [x1, y1, x2, y2], 'class_label': label}, ...]}
    """
    annotations = {}
    
    with open(annotations_file, 'r') as f:
        lines = f.readlines()
    
    i = 0
    num_poles = int(lines[i].strip())
    i += 1
    
    for pole_idx in range(num_poles):
        # Skip empty lines
        while i < len(lines) and lines[i].strip() == '':
            i += 1
        
        # Pole number and ID
        pole_info = lines[i].strip().split()
        i += 1
        
        # 3D coordinates
        i += 1
        
        # Number of traffic signs and their types
        i += 1
        
        # Number of centers
        num_centers = int(lines[i].strip())
        i += 1
        
        # Skip center coordinates
        for _ in range(num_centers):
            i += 1
        
        # Number of bounding boxes
        num_bboxes = int(lines[i].strip())
        i += 1
        
        # Parse each bounding box
        for _ in range(num_bboxes):
            # Bounding box coordinates
            bbox_coords = list(map(float, lines[i].strip().split()))
            # x1, y1, x2, y2 = bbox_coords
            y1, x1, y2, x2= bbox_coords
            i += 1
            
            # Camera, frame, order number
            metadata = lines[i].strip().split()
            camera = int(metadata[0])
            frame = int(metadata[1])
            i += 1
            
            # Class label
            class_label = lines[i].strip().rstrip(';')
            i += 1
            
            # 3D center coordinates
            i += 1
            
            # Create the key
            key = f"{camera:02d}/image.{frame:06d}.jp2"
            
            # Add annotation
            if key not in annotations:
                annotations[key] = []
            
            annotations[key].append({
                'bbox': [x1, y1, x2, y2],
                'class_label': class_label
            })
    
    return annotations

In [15]:
dataset_root = './data/belgiumts/sequences'
sequence_id = 2
sequence_path = f'Seq{sequence_id:02d}'
camera_id = 1
camera_id_str = f'{camera_id:02d}'
video_files_path = os.path.join(dataset_root, sequence_path, camera_id_str)
annotations_file = os.path.join(dataset_root, f'sequence{sequence_id}_GT.txt')

In [16]:
# Parse the annotations
annotations = parse_annotations(annotations_file)

# Display sample
print(f"Total unique images with annotations: {len(annotations)}")
print(f"\nSample annotations:")
for idx, (key, bboxes) in enumerate(list(annotations.items())[:3]):
    print(f"\n{key}: {len(bboxes)} annotations")
    for bbox_data in bboxes[:2]:  # Show first 2 bboxes per image
        print(f"  {bbox_data}")

camera_counts = {}

for key, bboxes in annotations.items():
    curr_camera_id = key.split('/')[0]
    if curr_camera_id not in camera_counts:
        camera_counts[curr_camera_id] = 0
    camera_counts[curr_camera_id] += len(bboxes)

print("Annotations per camera:")
for curr_camera_id in sorted(camera_counts.keys()):
    print(f"Camera {curr_camera_id}: {camera_counts[curr_camera_id]} annotations")

print(f"\nTotal annotations: {sum(camera_counts.values())}")

Total unique images with annotations: 279

Sample annotations:

01/image.001011.jp2: 1 annotations
  {'bbox': [1524.37, 497.96, 1601.67, 580.48], 'class_label': 'D9'}

01/image.001010.jp2: 1 annotations
  {'bbox': [1472.12, 479.35, 1545.06, 556.71], 'class_label': 'D9'}

01/image.001009.jp2: 1 annotations
  {'bbox': [1428.12, 461.82, 1495.88, 533.45], 'class_label': 'D9'}
Annotations per camera:
Camera 00: 8 annotations
Camera 01: 227 annotations
Camera 02: 135 annotations
Camera 04: 6 annotations
Camera 05: 63 annotations
Camera 06: 27 annotations
Camera 07: 29 annotations

Total annotations: 495


In [17]:
for frame_file in sorted(os.listdir(video_files_path)):
    image_id = f'{camera_id_str}/{frame_file}'
    frame_annotations = annotations.get(image_id, [])

    if not frame_annotations:
        # print(f'{frame_file} skipped')
        continue

    frame_path = os.path.join(video_files_path, frame_file)
    image = cv.imread(frame_path)

    img_height, img_width = image.shape[:2]

    for annotation in frame_annotations:
        x1_orig, y1_orig, x2_orig, y2_orig = map(int, annotation['bbox'])
        class_label = annotation['class_label']

        # Apply 90-degree clockwise rotation transformation
        # new_x = old_y
        # new_y = img_height - old_x
        x1 = x1_orig
        y1 = y2_orig
        x2 = x2_orig
        y2 = y1_orig
        
        # x1 = y1_orig
        # y1 = x2_orig
        # x2 = y2_orig
        # y2 = x1_orig

        # Mirror around x-axis (vertical flip)
        # y1 = img_height - y1
        # y2 = img_height - y2
        
        # print(f"Original: ({x1_orig}, {y1_orig}, {x2_orig}, {y2_orig}) -> Final: ({x1}, {y1}, {x2}, {y2}), Image shape: {image.shape}")
        image = cv.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 4)
        # image = cv.putText(image, class_label, (x1, y1 - 10), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    image = cv.resize(image, None, fx=0.7, fy=0.7)

    cv.imshow('Annotated Image', image)

    while True:
        key = cv.waitKey(0)
        if key == ord(' ') or key == ord('q'):
            break
    
    # key = cv.waitKey(10000)
    if key == ord('q'):
        break

cv.destroyAllWindows()